In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window

# Initialize Spark session
spark = SparkSession.builder.appName("OrdersDataFrame").getOrCreate()

# Define the dataset
data = [
    (1, 1, 100, "2023-04-01"),
    (2, 1, 200, "2023-04-02"),
    (3, 2, 300, "2023-04-03"),
    (4, 2, 300, "2023-04-04"),
    (5, 3, 500, "2023-04-05"),
    (6, 3, 400, "2023-04-06"),
    (7, 3, 600, "2023-04-07")
]

# Define schema
columns = ["order_id", "customer_id", "amount", "order_date"]

# Create DataFrame
orders_df = spark.createDataFrame(data, columns)

# Show DataFrame
orders_df.show()


In [0]:
result_df = (
    orders_df.withColumn(
        "rn",
        f.dense_rank().over(
            Window.partitionBy("customer_id").orderBy(
                f.desc("amount"), f.asc("order_date")
            )
        )
    )
    .withColumn(
        "order_cnt", f.count("order_id").over(Window.partitionBy("customer_id"))
    )
    .filter((f.col("rn") == 2) & (f.col("order_cnt") >= 2))
    .select(f.col("customer_id"), f.col("amount").alias("second_highest_amount"))
)
display(result_df)